# ML Modeling 

Baseline focus:
- `label_sensitive_binary`
- `label_target_main`

Split settings:
- `random` for in-sample capability
- `state` for cross-state generalization


## 1) Imports


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
import polars as pl

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, average_precision_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from sklearn.model_selection import StratifiedGroupKFold
except ImportError:
    StratifiedGroupKFold = None

try:
    import yaml
except ImportError as exc:
    raise ImportError('Please install pyyaml: pip install pyyaml') from exc

try:
    import joblib
except ImportError:
    joblib = None

## 2) Paths and Config


In [28]:
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
CFG_PATH = REPO_ROOT / 'config' / 'paths.local.yaml'
if not CFG_PATH.exists():
    CFG_PATH = REPO_ROOT / 'config' / 'paths.template.yaml'
if not CFG_PATH.exists():
    raise FileNotFoundError('Missing config/paths.local.yaml and config/paths.template.yaml')

cfg = yaml.safe_load(CFG_PATH.read_text(encoding='utf-8')) or {}
shared_cfg = cfg.get('shared', {})
repo_cfg = cfg.get('repo', {})

modeling_input = shared_cfg.get('modeling_input')
if not modeling_input:
    raise ValueError('Set shared.modeling_input in YAML')

DATA_PATH = Path(modeling_input)
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}')

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUNS_BASE_DIR = REPO_ROOT / repo_cfg.get('modeling_runs_dir', 'results/runs/modeling')
RUN_DIR = RUNS_BASE_DIR / f'run_{RUN_ID}'
MODEL_OBJECTS_DIR = REPO_ROOT / repo_cfg.get('model_objects_dir', 'results/runs/model_objects')

# Folders are created only when we actually save outputs/models.
SAVE_MODEL_OBJECTS = bool(repo_cfg.get('save_model_objects', False))

print('Config:', CFG_PATH)
print('Dataset:', DATA_PATH)
print('Run dir:', RUN_DIR)

Config: /share/project_repo/zzhan342/WiFiMLCapstoneS2026/config/paths.local.yaml
Dataset: /share/2026-01-17-10-50-world.tsv_chunks/world0_mlprep/world0_feb11_integrated_output_ml_dataset.parquet
Run dir: /share/project_repo/zzhan342/WiFiMLCapstoneS2026/results/runs/modeling/run_20260330T055154Z


## 3) Run Settings


In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42
MAX_ROWS = 2_000_000  # None = full dataset. Set int for smoke test.

TARGETS = ['label_sensitive_binary', 'label_target_main']
SPLIT_MODES = ['random', 'state']

STATE_N_SPLITS = 5
STATE_MIN_ROWS_PER_STATE = 5000
STATE_MAX_TRAIN_ROWS = 5_000_000  # For 'State' Split: Caps train size per state fold.

DROP_UNKNOWN_TARGET = True  # Drop 'unknown' from label_target_main — labeling artifact, not a real class.

CAT_COLS = ['vendor_clean']

#TODO: Update to other feature engineering method.
OHE_MAX_CATEGORIES = 50           # Rare vendors bundled into 'infrequent_category'.

RUN_RF = False  # Set True to include Random Forest.

_BASE_EXCLUDE = {'label_target_main', 'label_sensitive_binary', 'label_sensitive_subtype', 'add_state'}

# Ablation configs: (label, extra_cols_to_exclude).
# Compare full vs no_osm_scores in the summary table.
ABLATION_CONFIGS = [
    ('full',          set()),
    ('no_osm_scores', {'place_rank', 'importance'}),  # isolates pure Wi-Fi metadata signal
]

## 4) Load Dataset + Quick Checkpoints


In [ ]:
if DATA_PATH.suffix == '.parquet':
    pl_df = pl.read_parquet(str(DATA_PATH))
else:
    pl_df = pl.read_csv(str(DATA_PATH), separator='\t')

if MAX_ROWS is not None:
    pl_df = pl_df.sample(n=min(MAX_ROWS, pl_df.height), seed=RANDOM_STATE)

if 'add_state' in pl_df.columns:
    pl_df = pl_df.with_columns(pl.col('add_state').cast(pl.String).str.to_uppercase())

# Quick sanity check — full label distributions are in the mlprep summary files.
print('Shape:', pl_df.shape)
print(pl_df.head(3))

# Convert to pandas for sklearn. Drop Polars copy immediately to free memory.
pdf = pl_df.to_pandas()
del pl_df

## 4b) Feature Preview

In [33]:
print(f"All dataset columns ({len(pdf.columns)}): {list(pdf.columns)}")
print(f"Always excluded: {sorted(_BASE_EXCLUDE)}\n")

for ablation_label, extra_exclude in ABLATION_CONFIGS:
    exclude = _BASE_EXCLUDE | extra_exclude
    feature_cols = [c for c in pdf.columns if c not in exclude]
    cat_cols = [c for c in CAT_COLS if c in feature_cols]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    ohe_width = sum(min(pdf[c].nunique(), OHE_MAX_CATEGORIES) for c in cat_cols)
    total_width = len(num_cols) + ohe_width

    print(f"── feature_set='{ablation_label}' | {len(feature_cols)} raw cols → ~{total_width} after OHE ──")
    if extra_exclude:
        print(f"   additionally excluded: {sorted(extra_exclude)}")
    print(f"   numeric     ({len(num_cols)}): {num_cols}")
    print(f"   categorical ({len(cat_cols)}): {cat_cols}")
    for col in cat_cols:
        n_unique = pdf[col].nunique()
        top5 = pdf[col].value_counts().head(5).index.tolist()
        print(f"     {col}: {n_unique} unique values → {min(n_unique, OHE_MAX_CATEGORIES)} OHE cols  |  top 5: {top5}")
    print()

All dataset columns (8): ['add_state', 'place_rank', 'importance', 'altitude', 'vendor_clean', 'label_sensitive_binary', 'label_target_main', 'label_sensitive_subtype']
Always excluded: ['add_state', 'label_sensitive_binary', 'label_sensitive_subtype', 'label_target_main']

── feature_set='full' | 4 raw cols → ~53 after OHE ──
   numeric     (3): ['place_rank', 'importance', 'altitude']
   categorical (1): ['vendor_clean']
     vendor_clean: 1063 unique values → 50 OHE cols  |  top 5: ['NaN', 'Vantiva', 'eero inc.', 'Hewlett Packard', 'Cisco']

── feature_set='no_osm_scores' | 2 raw cols → ~51 after OHE ──
   additionally excluded: ['importance', 'place_rank']
   numeric     (1): ['altitude']
   categorical (1): ['vendor_clean']
     vendor_clean: 1063 unique values → 50 OHE cols  |  top 5: ['NaN', 'Vantiva', 'eero inc.', 'Hewlett Packard', 'Cisco']



## 5) Modeling Helpers

### Preprocessing
- **`StandardScaler(with_mean=False)`**: scales numeric features to unit std so LR treats all features equally; `with_mean=False` avoids converting the sparse OHE matrix to dense
- **`OneHotEncoder`**: expands `vendor_clean` into one 0/1 column per vendor.
- **`ColumnTransformer`**: applies numeric and categorical pipelines to their respective column groups, then combine outputs into one matrix for the classifier

### Split strategies
- **`random`**: stratified 80/20 — class ratios preserved; tests in-sample capability
- **`state`**: `StratifiedGroupKFold` — all records from the same state stay together; test folds contain only unseen states; 5 folds = train on 4/5 of states, test on 1/5; tests geographic generalization

### Helpers
- **`build_split_folds`**: returns `[(fold_id, train_idx, test_idx)]` for the chosen split strategy
- **`evaluate_model`**: accuracy, macro/weighted F1, confusion matrix, classification report; adds PR-AUC + ROC-AUC for binary targets
- **`save_run_artifacts`**: saves `summary_metrics.csv`, `run_metrics_compact.jsonl`, `run_metadata.json`; creates `RUN_DIR` on first call

In [34]:
def build_split_folds(y, split_mode, state_series=None):
    """Return list of (fold_id, train_idx, test_idx) for the given split strategy."""

    if split_mode == 'random':
        # Single stratified split — preserves class ratios in both halves.
        train_idx, test_idx = train_test_split(
            y.index, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
        )
        return [('random_fold_1', pd.Index(train_idx), pd.Index(test_idx))]

    if split_mode == 'state':
        if state_series is None:
            raise ValueError('state split requires state_series')

        groups = state_series.astype('string').str.upper().str.strip()
        valid_mask = groups.notna() & (groups != '')
        y_valid = y.loc[valid_mask]
        groups_valid = groups.loc[valid_mask]

        # Drop states with too few rows.
        state_counts = groups_valid.value_counts()
        eligible = set(state_counts[state_counts >= STATE_MIN_ROWS_PER_STATE].index)
        keep = groups_valid.isin(eligible)
        y_valid = y_valid.loc[keep]
        groups_valid = groups_valid.loc[keep]

        n_groups = int(groups_valid.nunique())
        if n_groups < 3:
            raise ValueError(f'Need >=3 eligible states for grouped CV, got {n_groups}')

        n_splits = min(STATE_N_SPLITS, n_groups)

        # X_dummy: GroupKFold API requires an X argument but only uses its row count.
        X_dummy = np.zeros((len(y_valid), 1))

        # StratifiedGroupKFold: grouped by state AND tries to balance class ratios per fold.
        # Fallback to GroupKFold for older sklearn versions.
        if StratifiedGroupKFold is not None and y_valid.nunique() > 1:
            splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
            split_iter = splitter.split(X_dummy, y_valid, groups_valid)
        else:
            splitter = GroupKFold(n_splits=n_splits)
            split_iter = splitter.split(X_dummy, y_valid, groups=groups_valid)

        folds = []
        y_idx = y_valid.index
        for i, (tr, te) in enumerate(split_iter, start=1):
            folds.append((f'state_fold_{i}', pd.Index(y_idx[tr]), pd.Index(y_idx[te])))

        print(f'  State split: {n_groups} eligible states → {n_splits} folds')
        return folds

    raise ValueError(f'Unknown split_mode: {split_mode}')

In [35]:
def evaluate_model(pipe, X_test, y_test):
    """Compute classification metrics. For binary targets also computes PR-AUC and ROC-AUC."""
    pred = pipe.predict(X_test)
    out = {
        'accuracy':    float(accuracy_score(y_test, pred)),
        # macro_f1: equal weight per class — fair to minority classes.
        'macro_f1':    float(f1_score(y_test, pred, average='macro',    zero_division=0)),
        # weighted_f1: weighted by class size — dominated by majority class.
        'weighted_f1': float(f1_score(y_test, pred, average='weighted', zero_division=0)),
        'confusion_matrix':      confusion_matrix(y_test, pred),
        'classification_report': classification_report(y_test, pred, output_dict=True, zero_division=0),
    }
    # PR-AUC is more informative than ROC-AUC under class imbalance.
    if len(pd.Series(y_test).dropna().unique()) == 2 and hasattr(pipe, 'predict_proba'):
        proba = pipe.predict_proba(X_test)[:, 1]
        out['pr_auc']  = float(average_precision_score(y_test, proba))
        out['roc_auc'] = float(roc_auc_score(y_test, proba))
    return out

In [36]:
def run_experiments(pdf):
    results = []

    for target_col in TARGETS:
        if target_col not in pdf.columns:
            print('Skip missing target:', target_col)
            continue

        work_df = pdf[pdf[target_col].notna()].copy()

        if target_col == 'label_target_main' and DROP_UNKNOWN_TARGET:
            before = len(work_df)
            work_df = work_df[work_df[target_col] != 'unknown'].copy()
            print(f'  Dropped {before - len(work_df):,} unknown rows from {target_col}')

        state_series = work_df['add_state'] if 'add_state' in work_df.columns else None
        y_full = work_df[target_col]
        print(f'\nTarget={target_col}, n={len(work_df):,}')

        for split_mode in SPLIT_MODES:
            folds = build_split_folds(y_full, split_mode=split_mode, state_series=state_series)

            for fold_id, train_idx, test_idx in folds:
                # State folds: cap train size so RF stays tractable. Test set always untouched.
                if split_mode == 'state' and STATE_MAX_TRAIN_ROWS and len(train_idx) > STATE_MAX_TRAIN_ROWS:
                    rng = np.random.default_rng(RANDOM_STATE)
                    train_idx = pd.Index(rng.choice(train_idx, size=STATE_MAX_TRAIN_ROWS, replace=False))

                for ablation_label, extra_exclude in ABLATION_CONFIGS:
                    exclude = _BASE_EXCLUDE | extra_exclude
                    feature_cols = [c for c in work_df.columns if c != target_col and c not in exclude]
                    X_train = work_df.loc[train_idx, feature_cols]
                    X_test  = work_df.loc[test_idx,  feature_cols]
                    y_train = y_full.loc[train_idx]
                    y_test  = y_full.loc[test_idx]

                    # Fresh model instances per (fold, ablation).
                    models = {
                        'logreg_balanced': LogisticRegression(max_iter=1000, class_weight='balanced'),
                    }
                    if RUN_RF:
                        models['rf_balanced'] = RandomForestClassifier(
                            n_estimators=300, random_state=RANDOM_STATE,
                            class_weight='balanced_subsample', n_jobs=-1,
                        )

                    for model_name, model in models.items():
                        # num_cols is dynamic: ablation may drop place_rank/importance from X_train.
                        num_cols = [c for c in X_train.columns if c not in CAT_COLS]
                        pipe = Pipeline([
                            ('preprocessor', ColumnTransformer([
                                # Numeric: scale to unit std (with_mean=False keeps OHE sparse).
                                ('num', StandardScaler(with_mean=False), num_cols),
                                # Categorical
                                ('cat', OneHotEncoder(
                                    handle_unknown='infrequent_if_exist',
                                    max_categories=OHE_MAX_CATEGORIES,
                                ), CAT_COLS),
                            ])),
                            ('classifier', model),
                        ])
                        pipe.fit(X_train, y_train)
                        metric = evaluate_model(pipe, X_test, y_test)

                        out = {
                            'target': target_col, 'split_mode': split_mode,
                            'fold_id': fold_id, 'feature_set': ablation_label,
                            'model': model_name,
                            'n_train': int(len(X_train)), 'n_test': int(len(X_test)),
                            **metric,
                        }
                        results.append(out)
                        print(
                            f"  {target_col} | {split_mode} | {fold_id} | {ablation_label} | {model_name}"
                            f" → macro_f1={out['macro_f1']:.4f}"
                        )

                        if SAVE_MODEL_OBJECTS and joblib is not None:
                            MODEL_OBJECTS_DIR.mkdir(parents=True, exist_ok=True)
                            fname = f"{RUN_ID}__{target_col}__{split_mode}__{fold_id}__{ablation_label}__{model_name}.joblib"
                            joblib.dump(pipe, MODEL_OBJECTS_DIR / fname)
    return results

In [37]:
def save_run_artifacts(results, summary):
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    # Main results table.
    summary.to_csv(RUN_DIR / 'summary_metrics.csv', index=False)
    # Per-run records with confusion matrices for later analysis.
    compact = []
    for r in results:
        rec = {k: v for k, v in r.items() if k not in ('confusion_matrix', 'classification_report')}
        rec['confusion_matrix'] = r['confusion_matrix'].tolist()
        compact.append(rec)
    with (RUN_DIR / 'run_metrics_compact.jsonl').open('w') as f:
        for rec in compact:
            f.write(json.dumps(rec) + '\n')
    # Config snapshot for reproducibility.
    json.dump({
        'run_id': RUN_ID, 'data_path': str(DATA_PATH),
        'targets': TARGETS, 'split_modes': SPLIT_MODES,
        'max_rows': MAX_ROWS, 'run_rf': RUN_RF,
        'state_n_splits': STATE_N_SPLITS, 'state_max_train_rows': STATE_MAX_TRAIN_ROWS,
        'drop_unknown_target': DROP_UNKNOWN_TARGET,
        'ohe_max_categories': OHE_MAX_CATEGORIES,
        'ablation_configs': [a for a, _ in ABLATION_CONFIGS],
    }, (RUN_DIR / 'run_metadata.json').open('w'), indent=2)
    print('Saved run artifacts to:', RUN_DIR)

## 6) Run Baseline Experiments


In [38]:
results = run_experiments(pdf)
print('Total runs:', len(results))



Target=label_sensitive_binary, n=2,000,000
  label_sensitive_binary | random | random_fold_1 | full | logreg_balanced → macro_f1=0.6122
  label_sensitive_binary | random | random_fold_1 | no_osm_scores | logreg_balanced → macro_f1=0.6086
  State split: 46 eligible states → 5 folds
  label_sensitive_binary | state | state_fold_1 | full | logreg_balanced → macro_f1=0.5969
  label_sensitive_binary | state | state_fold_1 | no_osm_scores | logreg_balanced → macro_f1=0.5957
  label_sensitive_binary | state | state_fold_2 | full | logreg_balanced → macro_f1=0.6032
  label_sensitive_binary | state | state_fold_2 | no_osm_scores | logreg_balanced → macro_f1=0.6175
  label_sensitive_binary | state | state_fold_3 | full | logreg_balanced → macro_f1=0.6093
  label_sensitive_binary | state | state_fold_3 | no_osm_scores | logreg_balanced → macro_f1=0.5973
  label_sensitive_binary | state | state_fold_4 | full | logreg_balanced → macro_f1=0.6209
  label_sensitive_binary | state | state_fold_4 | no_

/share/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  label_target_main | random | random_fold_1 | full | logreg_balanced → macro_f1=0.4488
  label_target_main | random | random_fold_1 | no_osm_scores | logreg_balanced → macro_f1=0.2159
  State split: 45 eligible states → 5 folds


/share/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  label_target_main | state | state_fold_1 | full | logreg_balanced → macro_f1=0.4424
  label_target_main | state | state_fold_1 | no_osm_scores | logreg_balanced → macro_f1=0.2057


...

  label_target_main | state | state_fold_3 | full | logreg_balanced → macro_f1=0.4508
  label_target_main | state | state_fold_3 | no_osm_scores | logreg_balanced → macro_f1=0.1967


/share/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  label_target_main | state | state_fold_4 | full | logreg_balanced → macro_f1=0.4463
  label_target_main | state | state_fold_4 | no_osm_scores | logreg_balanced → macro_f1=0.2095


/share/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  label_target_main | state | state_fold_5 | full | logreg_balanced → macro_f1=0.4459
  label_target_main | state | state_fold_5 | no_osm_scores | logreg_balanced → macro_f1=0.2078
Total runs: 24


## 7) Summary Table


In [39]:
summary = pd.DataFrame([
    {
        'target': r['target'],
        'split_mode': r['split_mode'],
        'fold_id': r.get('fold_id', 'na'),
        'feature_set': r.get('feature_set', 'full'),
        'model': r['model'],
        'n_train': r['n_train'],
        'n_test': r['n_test'],
        'accuracy': r['accuracy'],
        'macro_f1': r['macro_f1'],
        'weighted_f1': r['weighted_f1'],
        'pr_auc': r.get('pr_auc', np.nan),
        'roc_auc': r.get('roc_auc', np.nan),
    }
    for r in results
]).sort_values(['target', 'split_mode', 'feature_set', 'model', 'fold_id'])

print('Summary head:')
print(summary.head(20))
summary

Summary head:
                    target split_mode        fold_id    feature_set  \
0   label_sensitive_binary     random  random_fold_1           full   
1   label_sensitive_binary     random  random_fold_1  no_osm_scores   
2   label_sensitive_binary      state   state_fold_1           full   
4   label_sensitive_binary      state   state_fold_2           full   
6   label_sensitive_binary      state   state_fold_3           full   
8   label_sensitive_binary      state   state_fold_4           full   
10  label_sensitive_binary      state   state_fold_5           full   
3   label_sensitive_binary      state   state_fold_1  no_osm_scores   
5   label_sensitive_binary      state   state_fold_2  no_osm_scores   
7   label_sensitive_binary      state   state_fold_3  no_osm_scores   
9   label_sensitive_binary      state   state_fold_4  no_osm_scores   
11  label_sensitive_binary      state   state_fold_5  no_osm_scores   
12       label_target_main     random  random_fold_1           

,target,split_mode,fold_id,feature_set,model,n_train,n_test,accuracy,macro_f1,weighted_f1,pr_auc,roc_auc
0,label_sensitive_binary,random,random_fold_1,full,logreg_balanced,1600000,400000,0.808750,0.612191,0.858235,0.503942,0.927768
1,label_sensitive_binary,random,random_fold_1,no_osm_scores,logreg_balanced,1600000,400000,0.842997,0.608599,0.878526,0.246598,0.819000
2,label_sensitive_binary,state,state_fold_1,full,logreg_balanced,1573479,402305,0.796980,0.596850,0.851055,0.483969,0.921189
4,label_sensitive_binary,state,state_fold_2,full,logreg_balanced,1585193,390591,0.798352,0.603240,0.850818,0.512438,0.923615
6,label_sensitive_binary,state,state_fold_3,full,logreg_balanced,1580310,395474,0.807874,0.609349,0.857506,0.492019,0.922208
8,label_sensitive_binary,state,state_fold_4,full,logreg_balanced,1570507,405277,0.823160,0.620866,0.869118,0.509891,0.936507
10,label_sensitive_binary,state,state_fold_5,full,logreg_balanced,1593647,382137,0.800870,0.612515,0.851802,0.514833,0.929550
3,label_sensitive_binary,state,state_fold_1,no_osm_scores,logreg_balanced,1573479,402305,0.834108,0.595689,0.873547,0.230605,0.808910
5,label_sensitive_binary,state,state_fold_2,no_osm_scores,logreg_balanced,1585193,390591,0.854725,0.617453,0.885538,0.244829,0.808118
7,label_sensitive_binary,state,state_fold_3,no_osm_scores,logreg_balanced,1580310,395474,0.830310,0.597253,0.870259,0.247433,0.813048


## 8) Save Run Artifacts


In [40]:
save_run_artifacts(results, summary)

Saved run artifacts to: /share/project_repo/zzhan342/WiFiMLCapstoneS2026/results/runs/modeling/run_20260330T055154Z


## 9) Detailed Metrics Checkpoints

In [41]:
for r in results:
    print("\n" + "=" * 90)
    print(f"target={r['target']} | split={r['split_mode']} | fold={r['fold_id']} | features={r['feature_set']} | model={r['model']}")
    print('Confusion matrix:')
    print(r['confusion_matrix'])
    print()
    print(pd.DataFrame(r['classification_report']).T)


target=label_sensitive_binary | split=random | fold=random_fold_1 | features=full | model=logreg_balanced
Confusion matrix:
[[304136  74097]
 [  2403  19364]]

              precision    recall  f1-score       support
0              0.992161  0.804097  0.888284  378233.00000
1              0.207188  0.889604  0.336099   21767.00000
accuracy       0.808750  0.808750  0.808750       0.80875
macro avg      0.599674  0.846850  0.612191  400000.00000
weighted avg   0.949445  0.808750  0.858235  400000.00000

target=label_sensitive_binary | split=random | fold=random_fold_1 | features=no_osm_scores | model=logreg_balanced
Confusion matrix:
[[323373  54860]
 [  7941  13826]]

              precision    recall  f1-score        support
0              0.976032  0.854957  0.911491  378233.000000
1              0.201293  0.635182  0.305706   21767.000000
accuracy       0.842997  0.842997  0.842997       0.842997
macro avg      0.588662  0.745069  0.608599  400000.000000
weighted avg   0.933872  0